# 29 — Engineering Constraints Determine Usable Cluster-State Computation

**Engineering question**

How much of an ideal frequency-bin cluster state survives as usable computation once squeezing, loss, detection, control, and feed-forward constraints are included?

Notebook 23 described the cluster-state architecture.  
Notebook 29 asks what determines whether that architecture actually becomes useful:

```text
ideal cluster graph
→ finite squeezing
→ optical loss
→ detector inefficiency
→ mode-control errors
→ feed-forward latency
→ usable computational graph
```

Engineering statement:

> Finite squeezing, optical loss, and control determine how much of the ideal cluster state becomes usable computation.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/29_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Ideal graph versus usable graph

An ideal cluster state is a connected entanglement graph.

A real system has:

- lost nodes,
- weak edges,
- imperfect measurements,
- disconnected regions.

This figure illustrates the engineering difference between the graph that is designed and the graph that remains usable.


In [ ]:
rows, cols = 5, 7
nodes = [(i, j) for i in range(cols) for j in range(rows)]

lost_nodes = {(1, 3), (3, 1), (4, 4), (5, 2)}
weak_edges = {((2, 0), (3, 0)), ((2, 2), (2, 3)), ((5, 4), (6, 4)), ((0, 1), (1, 1))}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, title, degraded in zip(axes, ["Ideal cluster graph", "Usable graph after constraints"], [False, True]):
    ax.set_title(title, fontsize=14)
    ax.axis("off")
    ax.set_aspect("equal")

    # edges
    for i in range(cols):
        for j in range(rows):
            if i + 1 < cols:
                a, b = (i, j), (i + 1, j)
                if degraded and (a in lost_nodes or b in lost_nodes):
                    continue
                lw = 0.8 if degraded and ((a, b) in weak_edges or (b, a) in weak_edges) else 2.0
                alpha = 0.35 if lw < 1 else 0.95
                ax.plot([i, i + 1], [j, j], linewidth=lw, alpha=alpha)
            if j + 1 < rows:
                a, b = (i, j), (i, j + 1)
                if degraded and (a in lost_nodes or b in lost_nodes):
                    continue
                lw = 0.8 if degraded and ((a, b) in weak_edges or (b, a) in weak_edges) else 2.0
                alpha = 0.35 if lw < 1 else 0.95
                ax.plot([i, i], [j, j + 1], linewidth=lw, alpha=alpha)

    # nodes
    for i, j in nodes:
        if degraded and (i, j) in lost_nodes:
            ax.scatter([i], [j], s=130, marker="x", linewidth=2.5)
        else:
            ax.scatter([i], [j], s=105)

    if degraded:
        ax.text(cols / 2 - 0.5, -0.55, "lost nodes + weak edges reduce usable connectivity", ha="center", fontsize=11)
    else:
        ax.text(cols / 2 - 0.5, -0.55, "designed graph resource", ha="center", fontsize=11)

fig.suptitle("Architecture Quality Determines the Usable Computational Graph", fontsize=16)

savefig(fig, "29_ideal_vs_usable_graph.png")
plt.show()


## 2. Resource budget versus loss

Optical loss reduces the usable graph.

This toy model tracks three quantities as loss increases:

- ideal graph nodes,
- usable graph nodes,
- executable computation depth.

It is an illustrative engineering trend, not experimental data.


In [ ]:
loss = np.linspace(0, 0.35, 250)
ideal_nodes = np.full_like(loss, 100.0)
usable_nodes = ideal_nodes * np.exp(-3.8 * loss)
logical_depth = 50 * np.exp(-6.2 * loss)

fig, ax = plt.subplots(figsize=(8.5, 5.2))

ax.plot(100 * loss, ideal_nodes, linewidth=2.4, label="ideal graph nodes")
ax.plot(100 * loss, usable_nodes, linewidth=2.4, label="usable graph nodes")
ax.plot(100 * loss, logical_depth, linewidth=2.4, label="logical computation depth")

ax.set_title("Resource Budget Shrinks with Optical Loss", fontsize=16)
ax.set_xlabel("Optical loss (%)")
ax.set_ylabel("Relative resource")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(12, 74, "illustrative engineering trend", fontsize=10)

savefig(fig, "29_resource_budget_vs_loss.png")
plt.show()


## 3. Constraint waterfall

A frequency comb can start with many modes.

But the usable computational resource is filtered through multiple constraints:

```text
generated modes
→ squeezed modes
→ entangled graph
→ measured graph
→ feed-forward compatible graph
→ logical computation
```

Each step preserves only part of the previous resource.


In [ ]:
stages = [
    "Generated\nmodes",
    "Squeezed\nmodes",
    "Entangled\ngraph",
    "Measured\ngraph",
    "Feed-forward\ncompatible",
    "Logical\ncomputation",
]
values = np.array([100, 82, 66, 54, 43, 32])

fig, ax = plt.subplots(figsize=(10, 5.2))
x = np.arange(len(stages))

ax.bar(x, values)
ax.plot(x, values, marker="o", linewidth=2.0)

for xi, val in zip(x, values):
    ax.text(xi, val + 2.5, f"{val:.0f}", ha="center", fontsize=10)

for i in range(len(x) - 1):
    ax.annotate("", xy=(i + 0.6, values[i + 1]), xytext=(i + 0.4, values[i]),
                arrowprops=dict(arrowstyle="->", linewidth=1.4))

ax.set_title("Constraint Waterfall from Modes to Computation", fontsize=16)
ax.set_ylabel("Relative surviving resource")
ax.set_xticks(x)
ax.set_xticklabels(stages)
ax.set_ylim(0, 112)
ax.grid(True, axis="y", alpha=0.3)

savefig(fig, "29_constraint_waterfall.png")
plt.show()


## 4. Sensitivity ranking

Different constraints affect the usable graph differently.

This chart is a qualitative engineering priority ranking.

It is designed to support discussion, not to claim measured sensitivity.


In [ ]:
constraints = [
    "Finite squeezing",
    "Optical loss",
    "Detection efficiency",
    "Mode control",
    "Feed-forward latency",
    "Thermal drift",
]
scores = np.array([10, 9, 7.5, 6.5, 5.5, 4.5])

fig, ax = plt.subplots(figsize=(8.2, 4.8))
y = np.arange(len(constraints))[::-1]

ax.barh(y, scores)
ax.set_yticks(y)
ax.set_yticklabels(constraints)
ax.set_xlabel("Relative priority")
ax.set_title("Illustrative Sensitivity Ranking", fontsize=16)
ax.set_xlim(0, 11)
ax.grid(True, axis="x", alpha=0.3)

for yi, score in zip(y, scores):
    ax.text(score + 0.15, yi, f"{score:g}", va="center", fontsize=10)

savefig(fig, "29_sensitivity_ranking.png")
plt.show()


## 5. Constraint interaction map

The constraints are not independent.

Loss weakens edges.  
Weak squeezing lowers correlation strength.  
Detector inefficiency degrades measurement quality.  
Feed-forward latency limits adaptive computation.

This figure shows the dependency chain.


In [ ]:
labels = [
    "Finite\nsqueezing",
    "Optical\nloss",
    "Detector\nefficiency",
    "Mode\ncontrol",
    "Feed-forward\nlatency",
    "Usable\ncluster state",
]

pos = {
    0: (0.05, 0.70),
    1: (0.05, 0.30),
    2: (0.38, 0.70),
    3: (0.38, 0.30),
    4: (0.68, 0.50),
    5: (0.90, 0.50),
}

edges = [
    (0, 5),
    (1, 5),
    (2, 5),
    (3, 5),
    (4, 5),
    (1, 2),
    (3, 4),
]

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axis("off")

for a, b in edges:
    ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="->", linewidth=1.8))

for i, label in enumerate(labels):
    x0, y0 = pos[i]
    rect = plt.Rectangle((x0 - 0.07, y0 - 0.08), 0.14, 0.16, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x0, y0, label, ha="center", va="center", fontsize=9.5)

ax.set_title("Constraint Interaction Map", fontsize=16)
ax.set_xlim(-0.05, 1.02)
ax.set_ylim(0.1, 0.92)

savefig(fig, "29_constraint_interaction_map.png")
plt.show()


## 6. Complete photonic QC pipeline

This final pipeline connects the full repo.

It is intentionally long: the point is to show how far the architecture must travel from a pump laser to usable computation.


In [ ]:
labels = [
    "Pump",
    "High-Q\nresonator",
    "Kerr\nnonlinearity",
    "Four-wave\nmixing",
    "Frequency\ncomb",
    "Two-mode\nsqueezing",
    "Entanglement\ngraph",
    "Cluster\nstate",
    "Measurement",
    "Feed-forward",
    "Logical\ncomputation",
    "Engineering\nlimits",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(17, 3.4))

for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.42, -0.25), 0.84, 0.5, fill=False, linewidth=1.8)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=8.5)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.42, 0), arrowprops=dict(arrowstyle="->", linewidth=1.8))

ax.set_title("Complete Pipeline: From Microcomb Physics to Usable Computation", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.65, 0.72)
ax.axis("off")

savefig(fig, "29_complete_pipeline.png")
plt.show()


## 7. Engineering summary

Notebook 29 makes the practical point:

```text
ideal graph size
≠
usable computation
```

The difference is determined by engineering constraints.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Constraint": "Finite squeezing",
            "Physical effect": "Weaker quadrature correlations",
            "Computational consequence": "Lower logical fidelity",
            "Engineering priority": "Improve nonlinear interaction and squeezing level",
        },
        {
            "Constraint": "Optical loss",
            "Physical effect": "Node and edge degradation",
            "Computational consequence": "Graph fragmentation",
            "Engineering priority": "Reduce propagation and coupling loss",
        },
        {
            "Constraint": "Detection efficiency",
            "Physical effect": "Noisier quadrature measurements",
            "Computational consequence": "Reduced success probability",
            "Engineering priority": "Improve homodyne/readout efficiency",
        },
        {
            "Constraint": "Feed-forward latency",
            "Physical effect": "Delayed adaptive basis updates",
            "Computational consequence": "Reduced algorithm speed/depth",
            "Engineering priority": "Integrate electronics and control",
        },
        {
            "Constraint": "Mode control",
            "Physical effect": "Mode mixing and routing errors",
            "Computational consequence": "Incorrect graph connectivity",
            "Engineering priority": "Programmable photonics and stabilization",
        },
    ]
)

fig, ax = plt.subplots(figsize=(13.5, 4.0))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(8.8)
table.scale(1.05, 1.75)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Constraints Determine Usable Computation", fontsize=16, pad=20)

savefig(fig, "29_engineering_summary.png")

summary.to_csv(CSV / "29_summary.csv", index=False)
summary.to_json(JS / "29_summary.json", orient="records", indent=2)

plt.show()
summary


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "29_outputs.zip"

files_to_zip = [
    FIG / "29_ideal_vs_usable_graph.png",
    FIG / "29_resource_budget_vs_loss.png",
    FIG / "29_constraint_waterfall.png",
    FIG / "29_sensitivity_ranking.png",
    FIG / "29_constraint_interaction_map.png",
    FIG / "29_complete_pipeline.png",
    FIG / "29_engineering_summary.png",
    CSV / "29_summary.csv",
    JS / "29_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- An ideal cluster graph is not the same as a usable computational graph.
- Finite squeezing weakens graph edges.
- Optical loss removes usable modes and fragments connectivity.
- Detector inefficiency and feed-forward latency reduce measurement-based computational depth.
- The practical architecture is determined by how much of the frequency-bin cluster resource survives engineering constraints.

## Next notebook

**37 — Loss Characterization**

Notebook 29 identifies loss as a dominant engineering constraint.

Notebook 37 should ask how loss can be measured and separated into components:

```text
fiber coupling
→ facet loss
→ waveguide propagation
→ resonator internal loss
→ detector/readout efficiency
→ usable squeezing budget
```
